# BikeStores DWH — 02 ETL ke SQLite

Notebook ini merupakan tahap **kedua** setelah:

```text
BikeStores_DWH_01_Staging_Cleansing_v2.ipynb
```

Input notebook ini adalah data yang sudah lolos cleansing pada:

```text
data/staging/clean/
```

Tujuan:

```text
STAGING CLEAN
     │
     ▼
Build Dimensions
     │
     ├── dim_waktu
     ├── dim_toko
     └── dim_produk
     │
     ▼
Build Facts
     │
     ├── fakta_penjualan_bulanan
     └── fakta_pengiriman_bulanan
     │
     ▼
Validation
     │
     ▼
SQLite DWH
     │
     ▼
Database Validation + ETL Evidence
```

Strategi testing menggunakan **full reload**: tabel DWH dibuat ulang setiap run sehingga notebook aman dijalankan berulang tanpa menggandakan data.


## 00 — Setup

Notebook menggunakan Google Drive yang sama dengan notebook cleansing.

Target database:

```text
BikeStores_DWH/database/bikestores_dwh.db
```


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sqlite3
import pandas as pd
import numpy as np
import json
import os

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/BikeStores_DWH')
else:
    PROJECT_DIR = Path.cwd() / 'BikeStores_DWH'

STAGING_CLEAN_DIR = PROJECT_DIR / 'data' / 'staging' / 'clean'
DATABASE_DIR = PROJECT_DIR / 'database'
EVIDENCE_DIR = PROJECT_DIR / 'data' / 'evidence' / 'etl'
LOG_DIR = PROJECT_DIR / 'logs'

DATABASE_DIR.mkdir(parents=True, exist_ok=True)
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATABASE_DIR / 'bikestores_dwh.db'

print('PROJECT_DIR       :', PROJECT_DIR)
print('STAGING_CLEAN_DIR:', STAGING_CLEAN_DIR)
print('DB_PATH           :', DB_PATH)
print('EVIDENCE_DIR      :', EVIDENCE_DIR)


## 01 — Validasi Input Staging Clean

Notebook ETL **tidak membaca `data/raw/` secara langsung**.

File staging yang dibutuhkan:

```text
brands.csv
categories.csv
products.csv
stores.csv
orders.csv
order_items.csv
```

Jika file belum tersedia, jalankan notebook cleansing terlebih dahulu.


In [ ]:
REQUIRED_FILES = [
    'brands.csv',
    'categories.csv',
    'products.csv',
    'stores.csv',
    'orders.csv',
    'order_items.csv',
]

missing_files = [
    filename
    for filename in REQUIRED_FILES
    if not (STAGING_CLEAN_DIR / filename).exists()
]

if missing_files:
    raise FileNotFoundError(
        'Staging clean belum lengkap. Jalankan notebook 01 terlebih dahulu. '
        'File yang hilang: ' + ', '.join(missing_files)
    )

print('PASS: seluruh input ETL tersedia.')


In [ ]:
# Load staging clean
brands = pd.read_csv(STAGING_CLEAN_DIR / 'brands.csv')
categories = pd.read_csv(STAGING_CLEAN_DIR / 'categories.csv')
products = pd.read_csv(STAGING_CLEAN_DIR / 'products.csv')
stores = pd.read_csv(STAGING_CLEAN_DIR / 'stores.csv')
orders = pd.read_csv(
    STAGING_CLEAN_DIR / 'orders.csv',
    parse_dates=['order_date', 'required_date', 'shipped_date']
)
order_items = pd.read_csv(STAGING_CLEAN_DIR / 'order_items.csv')

source_summary = pd.DataFrame([
    {'table': 'brands', 'rows': len(brands)},
    {'table': 'categories', 'rows': len(categories)},
    {'table': 'products', 'rows': len(products)},
    {'table': 'stores', 'rows': len(stores)},
    {'table': 'orders', 'rows': len(orders)},
    {'table': 'order_items', 'rows': len(order_items)},
])

source_summary


# 02 — Build Dimensions

Urutan:

```text
dim_waktu
dim_toko
dim_produk
```

Dimensi dibuat terlebih dahulu karena fact table membutuhkan key dari dimensi.


## 02.1 — `dim_waktu`

Grain:

```text
1 row = 1 bulan
```

Key:

```text
id_waktu = YYYYMM
```


In [ ]:
dim_waktu = (
    orders[['order_date']]
    .dropna()
    .assign(
        id_waktu=lambda x:
            x['order_date'].dt.year * 100
            + x['order_date'].dt.month,
        tahun=lambda x: x['order_date'].dt.year,
        bulan=lambda x: x['order_date'].dt.month,
        nama_bulan=lambda x: x['order_date'].dt.month_name(),
        kuartal=lambda x: 'Q' + x['order_date'].dt.quarter.astype(str)
    )
    [['id_waktu', 'tahun', 'bulan', 'nama_bulan', 'kuartal']]
    .drop_duplicates()
    .sort_values('id_waktu')
    .reset_index(drop=True)
)

dim_waktu


## 02.2 — `dim_toko`

`id_toko` adalah surrogate key DWH.

`id_toko_oltp` menyimpan source key dari `stores.store_id`.


In [ ]:
dim_toko = (
    stores
    .sort_values('store_id')
    .reset_index(drop=True)
    .rename(columns={
        'store_id': 'id_toko_oltp',
        'store_name': 'nama_toko',
        'street': 'alamat',
        'city': 'kota',
        'zip_code': 'kode_pos',
        'phone': 'telepon',
    })
)

dim_toko.insert(
    0,
    'id_toko',
    np.arange(1, len(dim_toko) + 1, dtype=int)
)

dim_toko = dim_toko[
    [
        'id_toko',
        'id_toko_oltp',
        'nama_toko',
        'alamat',
        'kota',
        'state',
        'kode_pos',
        'telepon',
        'email',
    ]
]

dim_toko


## 02.3 — `dim_produk`

Sumber:

```text
products
   ├── brands
   └── categories
```

Join menggunakan `brand_id` dan `category_id`.


In [ ]:
dim_produk = (
    products
    .merge(
        brands[['brand_id', 'brand_name']],
        on='brand_id',
        how='left',
        validate='many_to_one'
    )
    .merge(
        categories[['category_id', 'category_name']],
        on='category_id',
        how='left',
        validate='many_to_one'
    )
    .sort_values('product_id')
    .reset_index(drop=True)
    .rename(columns={
        'product_id': 'id_produk_oltp',
        'product_name': 'nama_produk',
        'brand_name': 'merek',
        'category_name': 'kategori',
        'model_year': 'tahun_model',
        'list_price': 'harga_list',
    })
)

dim_produk.insert(
    0,
    'id_produk',
    np.arange(1, len(dim_produk) + 1, dtype=int)
)

dim_produk = dim_produk[
    [
        'id_produk',
        'id_produk_oltp',
        'nama_produk',
        'merek',
        'kategori',
        'tahun_model',
        'harga_list',
    ]
]

dim_produk


## 02.4 — Validasi Dimensions

Pemeriksaan:
- key DWH unik;
- source key unik;
- tidak ada hasil join yang hilang;
- dimensi waktu valid.


In [ ]:
dimension_validation = []

def add_dim_check(table, check, passed, detail=''):
    dimension_validation.append({
        'table': table,
        'check': check,
        'status': 'PASS' if passed else 'FAIL',
        'detail': detail
    })

add_dim_check(
    'dim_waktu',
    'id_waktu unique',
    dim_waktu['id_waktu'].is_unique
)
add_dim_check(
    'dim_waktu',
    'bulan 1..12',
    dim_waktu['bulan'].between(1, 12).all()
)
add_dim_check(
    'dim_toko',
    'id_toko unique',
    dim_toko['id_toko'].is_unique
)
add_dim_check(
    'dim_toko',
    'id_toko_oltp unique',
    dim_toko['id_toko_oltp'].is_unique
)
add_dim_check(
    'dim_toko',
    'row count = stores',
    len(dim_toko) == len(stores),
    f'{len(dim_toko)} vs {len(stores)}'
)
add_dim_check(
    'dim_produk',
    'id_produk unique',
    dim_produk['id_produk'].is_unique
)
add_dim_check(
    'dim_produk',
    'id_produk_oltp unique',
    dim_produk['id_produk_oltp'].is_unique
)
add_dim_check(
    'dim_produk',
    'row count = products',
    len(dim_produk) == len(products),
    f'{len(dim_produk)} vs {len(products)}'
)
add_dim_check(
    'dim_produk',
    'merek tidak NULL',
    dim_produk['merek'].notna().all()
)
add_dim_check(
    'dim_produk',
    'kategori tidak NULL',
    dim_produk['kategori'].notna().all()
)

dimension_validation = pd.DataFrame(dimension_validation)
dimension_validation


In [ ]:
if (dimension_validation['status'] != 'PASS').any():
    raise ValueError(
        'Validasi dimensi gagal. Jangan lanjut ke fact table.'
    )

print('PASS: seluruh validasi dimensi berhasil.')


# 03 — Build `fakta_penjualan_bulanan`

Business rule:

```text
order_status = 4
```

Grain:

```text
1 bulan × 1 toko × 1 produk
```


In [ ]:
sales_detail = (
    orders.loc[orders['order_status'] == 4]
    .merge(
        order_items,
        on='order_id',
        how='inner',
        validate='one_to_many'
    )
)

sales_detail['id_waktu'] = (
    sales_detail['order_date'].dt.year * 100
    + sales_detail['order_date'].dt.month
)

sales_detail = (
    sales_detail
    .merge(
        dim_toko[['id_toko', 'id_toko_oltp']],
        left_on='store_id',
        right_on='id_toko_oltp',
        how='left',
        validate='many_to_one'
    )
    .merge(
        dim_produk[['id_produk', 'id_produk_oltp']],
        left_on='product_id',
        right_on='id_produk_oltp',
        how='left',
        validate='many_to_one'
    )
)

sales_detail['gross_value'] = (
    sales_detail['quantity']
    * sales_detail['list_price']
)

sales_detail['discount_value'] = (
    sales_detail['gross_value']
    * sales_detail['discount']
)

sales_detail['net_value'] = (
    sales_detail['gross_value']
    - sales_detail['discount_value']
)


In [ ]:
fakta_penjualan_bulanan = (
    sales_detail
    .groupby(
        ['id_waktu', 'id_toko', 'id_produk'],
        as_index=False
    )
    .agg(
        jumlah_order=('order_id', 'nunique'),
        total_qty=('quantity', 'sum'),
        total_penjualan_kotor=('gross_value', 'sum'),
        total_diskon=('discount_value', 'sum'),
        total_penjualan_bersih=('net_value', 'sum'),
    )
    .sort_values(['id_waktu', 'id_toko', 'id_produk'])
    .reset_index(drop=True)
)

# Kurangi noise floating point.
money_cols = [
    'total_penjualan_kotor',
    'total_diskon',
    'total_penjualan_bersih',
]

fakta_penjualan_bulanan[money_cols] = (
    fakta_penjualan_bulanan[money_cols].round(4)
)

fakta_penjualan_bulanan.insert(
    0,
    'id_fakta_penjualan',
    np.arange(
        1,
        len(fakta_penjualan_bulanan) + 1,
        dtype=int
    )
)

fakta_penjualan_bulanan.head()


# 04 — Build `fakta_pengiriman_bulanan`

Grain:

```text
1 toko × 1 bulan berdasarkan order_date
```

Late shipment:

```text
shipped_date > required_date
```

NULL `shipped_date` tetap valid untuk order tertentu dan tidak otomatis dianggap terlambat.


In [ ]:
shipping_detail = orders.copy()

shipping_detail['id_waktu'] = (
    shipping_detail['order_date'].dt.year * 100
    + shipping_detail['order_date'].dt.month
)

shipping_detail = shipping_detail.merge(
    dim_toko[['id_toko', 'id_toko_oltp']],
    left_on='store_id',
    right_on='id_toko_oltp',
    how='left',
    validate='many_to_one'
)

shipping_detail['flag_pending'] = (
    shipping_detail['order_status'] == 1
).astype(int)

shipping_detail['flag_diproses'] = (
    shipping_detail['order_status'] == 2
).astype(int)

shipping_detail['flag_ditolak'] = (
    shipping_detail['order_status'] == 3
).astype(int)

shipping_detail['flag_selesai'] = (
    shipping_detail['order_status'] == 4
).astype(int)

shipping_detail['flag_terlambat'] = (
    shipping_detail['shipped_date'].notna()
    & (
        shipping_detail['shipped_date']
        > shipping_detail['required_date']
    )
).astype(int)

shipping_detail['hari_proses'] = (
    shipping_detail['shipped_date']
    - shipping_detail['order_date']
).dt.days


In [ ]:
fakta_pengiriman_bulanan = (
    shipping_detail
    .groupby(
        ['id_waktu', 'id_toko'],
        as_index=False
    )
    .agg(
        jumlah_order=('order_id', 'count'),
        jumlah_order_pending=('flag_pending', 'sum'),
        jumlah_order_diproses=('flag_diproses', 'sum'),
        jumlah_order_ditolak=('flag_ditolak', 'sum'),
        jumlah_order_selesai=('flag_selesai', 'sum'),
        jumlah_order_terlambat=('flag_terlambat', 'sum'),
        rata_rata_hari_proses=('hari_proses', 'mean'),
    )
    .sort_values(['id_waktu', 'id_toko'])
    .reset_index(drop=True)
)

fakta_pengiriman_bulanan[
    'rata_rata_hari_proses'
] = fakta_pengiriman_bulanan[
    'rata_rata_hari_proses'
].round(4)

fakta_pengiriman_bulanan.insert(
    0,
    'id_fakta_pengiriman',
    np.arange(
        1,
        len(fakta_pengiriman_bulanan) + 1,
        dtype=int
    )
)

fakta_pengiriman_bulanan.head()


# 05 — Validasi Fact Tables

Fact belum boleh dimuat ke SQLite bila validasi ini gagal.


In [ ]:
fact_validation = []

def add_fact_check(table, check, passed, detail=''):
    fact_validation.append({
        'table': table,
        'check': check,
        'status': 'PASS' if passed else 'FAIL',
        'detail': detail
    })

# Penjualan: grain
sales_grain_dupes = int(
    fakta_penjualan_bulanan.duplicated(
        subset=['id_waktu', 'id_toko', 'id_produk']
    ).sum()
)

add_fact_check(
    'fakta_penjualan_bulanan',
    'grain unique',
    sales_grain_dupes == 0,
    f'duplicate grain = {sales_grain_dupes}'
)

# Penjualan: FK tidak NULL
sales_fk_null = int(
    fakta_penjualan_bulanan[
        ['id_waktu', 'id_toko', 'id_produk']
    ].isna().any(axis=1).sum()
)

add_fact_check(
    'fakta_penjualan_bulanan',
    'FK tidak NULL',
    sales_fk_null == 0,
    f'rows = {sales_fk_null}'
)

# Penjualan: gross - discount = net
reconciliation_ok = np.allclose(
    fakta_penjualan_bulanan['total_penjualan_kotor']
    - fakta_penjualan_bulanan['total_diskon'],
    fakta_penjualan_bulanan['total_penjualan_bersih'],
    atol=1e-3
)

add_fact_check(
    'fakta_penjualan_bulanan',
    'gross - discount = net',
    reconciliation_ok
)

# Pengiriman: grain
shipping_grain_dupes = int(
    fakta_pengiriman_bulanan.duplicated(
        subset=['id_waktu', 'id_toko']
    ).sum()
)

add_fact_check(
    'fakta_pengiriman_bulanan',
    'grain unique',
    shipping_grain_dupes == 0,
    f'duplicate grain = {shipping_grain_dupes}'
)

# Pengiriman: status reconciliation
status_sum = (
    fakta_pengiriman_bulanan['jumlah_order_pending']
    + fakta_pengiriman_bulanan['jumlah_order_diproses']
    + fakta_pengiriman_bulanan['jumlah_order_ditolak']
    + fakta_pengiriman_bulanan['jumlah_order_selesai']
)

status_reconciliation_ok = (
    fakta_pengiriman_bulanan['jumlah_order']
    == status_sum
).all()

add_fact_check(
    'fakta_pengiriman_bulanan',
    'jumlah order = sum status',
    status_reconciliation_ok
)

# Pengiriman: total order kembali ke source
add_fact_check(
    'fakta_pengiriman_bulanan',
    'total order = source',
    int(fakta_pengiriman_bulanan['jumlah_order'].sum())
    == len(orders),
    (
        f"{int(fakta_pengiriman_bulanan['jumlah_order'].sum())} "
        f"vs {len(orders)}"
    )
)

# Total late shipment
late_source = int(
    (
        orders['shipped_date'].notna()
        & (orders['shipped_date'] > orders['required_date'])
    ).sum()
)

late_fact = int(
    fakta_pengiriman_bulanan[
        'jumlah_order_terlambat'
    ].sum()
)

add_fact_check(
    'fakta_pengiriman_bulanan',
    'late shipment = source',
    late_fact == late_source,
    f'{late_fact} vs {late_source}'
)

fact_validation = pd.DataFrame(fact_validation)
fact_validation


In [ ]:
if (fact_validation['status'] != 'PASS').any():
    raise ValueError(
        'Validasi fact gagal. Database tidak akan dimuat.'
    )

print('PASS: seluruh validasi fact berhasil.')


# 06 — Buat Schema SQLite

SQLite digunakan sebagai target testing.

Setiap run:
1. fact table lama di-drop;
2. dimension table lama di-drop;
3. schema dibuat ulang;
4. data hasil ETL dimuat kembali.

Dengan strategi ini, testing bersifat **idempotent** dan tidak menggandakan data.


In [ ]:
schema_sql = '''
PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS fakta_pengiriman_bulanan;
DROP TABLE IF EXISTS fakta_penjualan_bulanan;
DROP TABLE IF EXISTS dim_produk;
DROP TABLE IF EXISTS dim_toko;
DROP TABLE IF EXISTS dim_waktu;

CREATE TABLE dim_waktu (
    id_waktu INTEGER PRIMARY KEY,
    tahun INTEGER NOT NULL,
    bulan INTEGER NOT NULL CHECK (bulan BETWEEN 1 AND 12),
    nama_bulan TEXT NOT NULL,
    kuartal TEXT NOT NULL CHECK (kuartal IN ('Q1','Q2','Q3','Q4')),
    UNIQUE (tahun, bulan)
);

CREATE TABLE dim_toko (
    id_toko INTEGER PRIMARY KEY,
    id_toko_oltp INTEGER NOT NULL UNIQUE,
    nama_toko TEXT NOT NULL,
    alamat TEXT,
    kota TEXT,
    state TEXT,
    kode_pos TEXT,
    telepon TEXT,
    email TEXT
);

CREATE TABLE dim_produk (
    id_produk INTEGER PRIMARY KEY,
    id_produk_oltp INTEGER NOT NULL UNIQUE,
    nama_produk TEXT NOT NULL,
    merek TEXT NOT NULL,
    kategori TEXT NOT NULL,
    tahun_model INTEGER,
    harga_list REAL NOT NULL CHECK (harga_list >= 0)
);

CREATE TABLE fakta_penjualan_bulanan (
    id_fakta_penjualan INTEGER PRIMARY KEY,
    id_waktu INTEGER NOT NULL,
    id_toko INTEGER NOT NULL,
    id_produk INTEGER NOT NULL,
    jumlah_order INTEGER NOT NULL CHECK (jumlah_order >= 0),
    total_qty INTEGER NOT NULL CHECK (total_qty >= 0),
    total_penjualan_kotor REAL NOT NULL CHECK (total_penjualan_kotor >= 0),
    total_diskon REAL NOT NULL CHECK (total_diskon >= 0),
    total_penjualan_bersih REAL NOT NULL CHECK (total_penjualan_bersih >= 0),

    FOREIGN KEY (id_waktu) REFERENCES dim_waktu(id_waktu),
    FOREIGN KEY (id_toko) REFERENCES dim_toko(id_toko),
    FOREIGN KEY (id_produk) REFERENCES dim_produk(id_produk),

    UNIQUE (id_waktu, id_toko, id_produk)
);

CREATE TABLE fakta_pengiriman_bulanan (
    id_fakta_pengiriman INTEGER PRIMARY KEY,
    id_waktu INTEGER NOT NULL,
    id_toko INTEGER NOT NULL,
    jumlah_order INTEGER NOT NULL CHECK (jumlah_order >= 0),
    jumlah_order_pending INTEGER NOT NULL CHECK (jumlah_order_pending >= 0),
    jumlah_order_diproses INTEGER NOT NULL CHECK (jumlah_order_diproses >= 0),
    jumlah_order_ditolak INTEGER NOT NULL CHECK (jumlah_order_ditolak >= 0),
    jumlah_order_selesai INTEGER NOT NULL CHECK (jumlah_order_selesai >= 0),
    jumlah_order_terlambat INTEGER NOT NULL CHECK (jumlah_order_terlambat >= 0),
    rata_rata_hari_proses REAL,

    FOREIGN KEY (id_waktu) REFERENCES dim_waktu(id_waktu),
    FOREIGN KEY (id_toko) REFERENCES dim_toko(id_toko),

    UNIQUE (id_waktu, id_toko),

    CHECK (
        jumlah_order =
        jumlah_order_pending
        + jumlah_order_diproses
        + jumlah_order_ditolak
        + jumlah_order_selesai
    )
);
'''

conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON;')
conn.executescript(schema_sql)

print('Schema SQLite berhasil dibuat.')


# 07 — Load ke SQLite

Urutan load:

```text
dim_waktu
   ↓
dim_toko
   ↓
dim_produk
   ↓
fakta_penjualan_bulanan
   ↓
fakta_pengiriman_bulanan
```


In [ ]:
# Simpan dengan append karena schema sudah dibuat manual beserta constraint.
dim_waktu.to_sql(
    'dim_waktu',
    conn,
    if_exists='append',
    index=False
)

dim_toko.to_sql(
    'dim_toko',
    conn,
    if_exists='append',
    index=False
)

dim_produk.to_sql(
    'dim_produk',
    conn,
    if_exists='append',
    index=False
)

fakta_penjualan_bulanan.to_sql(
    'fakta_penjualan_bulanan',
    conn,
    if_exists='append',
    index=False
)

fakta_pengiriman_bulanan.to_sql(
    'fakta_pengiriman_bulanan',
    conn,
    if_exists='append',
    index=False
)

conn.commit()

print('PASS: seluruh tabel berhasil dimuat ke SQLite.')


# 08 — Validasi Database

Validasi dilakukan kembali **setelah data sudah berada di SQLite**.

Ini penting karena validasi Pandas saja belum membuktikan bahwa hasil load database benar.


In [ ]:
database_validation = []

def sql_scalar(query):
    return conn.execute(query).fetchone()[0]

expected_counts = {
    'dim_waktu': len(dim_waktu),
    'dim_toko': len(dim_toko),
    'dim_produk': len(dim_produk),
    'fakta_penjualan_bulanan': len(fakta_penjualan_bulanan),
    'fakta_pengiriman_bulanan': len(fakta_pengiriman_bulanan),
}

for table, expected in expected_counts.items():
    actual = sql_scalar(f'SELECT COUNT(*) FROM {table}')
    database_validation.append({
        'check': f'row count {table}',
        'expected': expected,
        'actual': actual,
        'status': 'PASS' if actual == expected else 'FAIL'
    })

# SQLite FK integrity check
fk_errors = conn.execute(
    'PRAGMA foreign_key_check;'
).fetchall()

database_validation.append({
    'check': 'PRAGMA foreign_key_check',
    'expected': 0,
    'actual': len(fk_errors),
    'status': 'PASS' if len(fk_errors) == 0 else 'FAIL'
})

# Rekonsiliasi order
sql_total_orders = sql_scalar(
    '''
    SELECT SUM(jumlah_order)
    FROM fakta_pengiriman_bulanan
    '''
)

database_validation.append({
    'check': 'total order fact pengiriman',
    'expected': len(orders),
    'actual': sql_total_orders,
    'status': (
        'PASS'
        if sql_total_orders == len(orders)
        else 'FAIL'
    )
})

# Late shipment
sql_late = sql_scalar(
    '''
    SELECT SUM(jumlah_order_terlambat)
    FROM fakta_pengiriman_bulanan
    '''
)

database_validation.append({
    'check': 'late shipment',
    'expected': late_source,
    'actual': sql_late,
    'status': 'PASS' if sql_late == late_source else 'FAIL'
})

database_validation = pd.DataFrame(database_validation)
database_validation


In [ ]:
if (database_validation['status'] != 'PASS').any():
    raise ValueError(
        'Validasi database gagal. Periksa hasil sebelum melanjutkan.'
    )

print('PASS: seluruh validasi database berhasil.')


## 08.1 — Contoh Query Analitik

Query ini bukan dashboard. Tujuannya hanya memastikan DWH sudah dapat digunakan untuk analisis.


In [ ]:
query_sales_by_month = '''
SELECT
    w.tahun,
    w.bulan,
    w.nama_bulan,
    ROUND(SUM(f.total_penjualan_bersih), 2)
        AS total_penjualan_bersih
FROM fakta_penjualan_bulanan AS f
JOIN dim_waktu AS w
    ON f.id_waktu = w.id_waktu
GROUP BY
    w.tahun,
    w.bulan,
    w.nama_bulan
ORDER BY
    w.tahun,
    w.bulan;
'''

pd.read_sql_query(
    query_sales_by_month,
    conn
).head(12)


In [ ]:
query_store_performance = '''
SELECT
    t.nama_toko,
    SUM(f.jumlah_order) AS jumlah_order,
    SUM(f.jumlah_order_terlambat)
        AS jumlah_order_terlambat,
    ROUND(
        AVG(f.rata_rata_hari_proses),
        2
    ) AS rata_rata_hari_proses_bulanan
FROM fakta_pengiriman_bulanan AS f
JOIN dim_toko AS t
    ON f.id_toko = t.id_toko
GROUP BY
    t.id_toko,
    t.nama_toko
ORDER BY
    jumlah_order DESC;
'''

pd.read_sql_query(
    query_store_performance,
    conn
)


# 09 — Simpan Evidence dan ETL Log

Evidence disimpan pada:

```text
data/evidence/etl/
```

Minimal berisi:
- source summary;
- validation dimensions;
- validation facts;
- validation database;
- row count output;
- ETL run log.


In [ ]:
output_summary = pd.DataFrame([
    {'table': 'dim_waktu', 'rows': len(dim_waktu)},
    {'table': 'dim_toko', 'rows': len(dim_toko)},
    {'table': 'dim_produk', 'rows': len(dim_produk)},
    {
        'table': 'fakta_penjualan_bulanan',
        'rows': len(fakta_penjualan_bulanan)
    },
    {
        'table': 'fakta_pengiriman_bulanan',
        'rows': len(fakta_pengiriman_bulanan)
    },
])

source_summary.to_csv(
    EVIDENCE_DIR / 'etl_source_summary.csv',
    index=False
)

output_summary.to_csv(
    EVIDENCE_DIR / 'etl_output_summary.csv',
    index=False
)

dimension_validation.to_csv(
    EVIDENCE_DIR / 'dimension_validation.csv',
    index=False
)

fact_validation.to_csv(
    EVIDENCE_DIR / 'fact_validation.csv',
    index=False
)

database_validation.to_csv(
    EVIDENCE_DIR / 'database_validation.csv',
    index=False
)

run_id = datetime.now(timezone.utc).strftime(
    '%Y%m%dT%H%M%SZ'
)

etl_run_log = pd.DataFrame([
    {
        'run_id': run_id,
        'run_time_utc': datetime.now(timezone.utc).isoformat(),
        'database': str(DB_PATH),
        'dim_waktu_rows': len(dim_waktu),
        'dim_toko_rows': len(dim_toko),
        'dim_produk_rows': len(dim_produk),
        'fact_sales_rows': len(fakta_penjualan_bulanan),
        'fact_shipping_rows': len(fakta_pengiriman_bulanan),
        'status': 'SUCCESS'
    }
])

etl_run_log.to_csv(
    LOG_DIR / f'etl_run_{run_id}.csv',
    index=False
)

print('Evidence :', EVIDENCE_DIR)
print('Log      :', LOG_DIR)
print('Database :', DB_PATH)


# 10 — Expected Result untuk Dataset Proyek

Dengan dataset BikeStores yang digunakan pada proyek ini, hasil utama yang diharapkan adalah sekitar:

```text
dim_waktu                  = 35 rows
dim_toko                   = 3 rows
dim_produk                 = 321 rows
fakta_penjualan_bulanan    = 2,448 rows
fakta_pengiriman_bulanan   = 93 rows
```

Rekonsiliasi penting:

```text
total order pada fakta pengiriman
= 1,615

jumlah late shipment
= 458
```

Nilai tersebut berfungsi sebagai **sanity check** untuk versi dataset saat ini. Validasi utama tetap menggunakan source yang sedang dibaca notebook, bukan hard-coded angka tersebut.


# 11 — Kriteria Berhasil

Notebook dinyatakan berhasil jika:

```text
✓ seluruh staging clean ditemukan
✓ seluruh dimension validation = PASS
✓ seluruh fact validation = PASS
✓ schema SQLite berhasil dibuat
✓ seluruh tabel berhasil diload
✓ PRAGMA foreign_key_check = 0 error
✓ row count database sesuai dataframe
✓ total order direkonsiliasi ke source
✓ late shipment direkonsiliasi ke source
✓ evidence dan log berhasil disimpan
```

Jika notebook **Run all** selesai tanpa error dan semua pemeriksaan menunjukkan `PASS`, tahap implementasi ETL SQLite berhasil.

Tahap berikutnya setelah hasil ini diaudit adalah:

```text
09 — Validation & Testing
```

Pada tahap tersebut kita akan melakukan validasi yang lebih ketat, termasuk rekonsiliasi nilai penjualan dan pengujian rerun/idempotency.


In [ ]:
# Tutup koneksi setelah seluruh proses selesai.
conn.close()

print('Koneksi SQLite ditutup.')
print('ETL selesai.')
